# Notebook 07 — Jean et al. (2016) Transfer Learning Pipeline for Poverty Estimation

## What this notebook implements

This notebook implements the core methodology from:

> **Jean et al. (2016, *Science*)** — *Combining satellite imagery and machine learning to predict poverty.*

The pipeline has two stages:

```
Stage 1 — Transfer learning:
  Daytime satellite imagery → CNN trained to predict nighttime light (NTL) intensity
  (NTL is a cheap, globally available proxy for ground-truth wealth)

Stage 2 — Poverty prediction:
  CNN penultimate-layer features → Ridge regression → DHS survey wealth index
  (Generalises to tiles with NO ground truth → continental poverty maps)
```

This is the foundation of the PhD project at Chalmers (REF 2026-0232):
*"Doctoral student in Earth Observation, Data Science, and AI for poverty estimation"*

The three PhD objectives map directly to this notebook:
- **Objective 1**: deep-learning models for poverty estimation from Sentinel-2 → Stage 1 here
- **Objective 2**: comparing satellite resolutions → swap the imagery source in Stage 1
- **Objective 3**: XAI for model trust → GradCAM on Stage 1 CNN (see Notebook 04)

## Data used
We use **synthetic data** that mirrors real-world structure (NTL-wealth correlations,
spatial autocorrelation, urban morphology). Each synthetic component is documented with
exact instructions for swapping in real data (Sentinel-2, VIIRS, DHS surveys).

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)
np.random.seed(42)
print('Imports OK')

## Part 1 — Obtaining real data (instructions)

### Sentinel-2 over Africa
```python
# Option A: Copernicus Open Access Hub (free, no quota)
# https://scihub.copernicus.eu — search by bounding box and date
# Download L2A products (surface reflectance, atmospherically corrected)

# Option B: Google Earth Engine (free academic quota)
import ee
ee.Initialize()
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2022-01-01', '2022-12-31') \
    .filterBounds(ee.Geometry.BBox(33.0, 3.4, 48.0, 15.0))  # Ethiopia bbox \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .median()
```

### VIIRS nighttime lights (no account needed)
```python
# Download annual composite from Colorado School of Mines EOG:
# https://eogdata.mines.edu/products/vnl/
# File: VNL_v21_npp_2022_global_vcmslcfg_c202205302300.average_masked.dat.tif.gz
import rasterio, numpy as np
with rasterio.open('viirs_2022.tif') as src:
    ntl_patch = src.read(1)  # radiance in nW/cm^2/sr
```

### DHS survey wealth data (free academic access)
```
https://dhsprogram.com/data/
Register → request country datasets → download GPS clusters + wealth index
Countries with 2018-2022 surveys: Ethiopia, Ghana, Nigeria, Tanzania, Kenya, Uganda
```

### African DHS survey coverage
| Country | Survey year | Clusters | Bbox |
|---|---|---|---|
| Nigeria | 2018 | 1400 | (2.7, 4.3, 14.7, 13.9) |
| Ethiopia | 2019 | 645 | (33.0, 3.4, 48.0, 15.0) |
| Ghana | 2022 | 550 | (-3.3, 4.5, 1.2, 11.2) |
| Tanzania | 2022 | 608 | (29.3, -11.7, 40.4, -0.9) |

In [ ]:
# ── SYNTHETIC DATA — mimics Nigeria DHS + VIIRS + Sentinel-2 features ────────
# To use real data: replace ntl, wealth_index, image_features below.
#
# Spatial structure: 3 urban centres (Lagos/Kano/Abuja approximate) + rural.
# NTL-wealth correlation r≈0.65 matches the literature value.
# ─────────────────────────────────────────────────────────────────────────────

N_TILES  = 500    # DHS survey clusters
FEAT_DIM = 512    # ResNet-50 penultimate layer size

# Nigeria bounding box (real coordinates)
LON_MIN, LON_MAX = 2.7, 14.7
LAT_MIN, LAT_MAX = 4.3, 13.9

# Major city centroids: Lagos, Kano, Abuja
CITIES = [(3.4, 6.5, 60), (8.5, 12.0, 45), (7.5, 9.1, 50)]  # lon, lat, ntl_peak

lons = np.random.uniform(LON_MIN, LON_MAX, N_TILES)
lats = np.random.uniform(LAT_MIN, LAT_MAX, N_TILES)

# NTL: Gaussian decay from cities + rural noise
ntl = np.zeros(N_TILES)
for city_lon, city_lat, peak in CITIES:
    dist = np.sqrt((lons - city_lon)**2 + (lats - city_lat)**2)
    ntl += peak * np.exp(-dist**2 / (1.5**2))
ntl += np.random.exponential(1.5, N_TILES)
ntl = np.clip(ntl, 0, 80)

# DHS wealth index: correlated with NTL (r≈0.65) + independent variance
wealth_index = (
    0.55 * (ntl / ntl.max())
    + 0.45 * np.random.normal(0, 1, N_TILES)
)
wealth_index = (wealth_index - wealth_index.min()) / (wealth_index.max() - wealth_index.min())

# CNN features: 512-dim vectors encoding NTL-predictive visual patterns
# Urban tiles → features align with "urban texture" direction in feature space
urban_direction = np.random.randn(FEAT_DIM)
urban_direction /= np.linalg.norm(urban_direction)

ntl_norm = ntl / ntl.max()
image_features = (
    np.outer(ntl_norm, urban_direction) * 3.0   # NTL-correlated component
    + np.random.randn(N_TILES, FEAT_DIM) * 1.0  # irreducible noise
)

print(f"Region:           Nigeria ({LON_MIN}°E–{LON_MAX}°E, {LAT_MIN}°N–{LAT_MAX}°N)")
print(f"Survey clusters:  {N_TILES}")
print(f"NTL range:        {ntl.min():.1f} – {ntl.max():.1f} nW/cm²/sr")
print(f"DHS wealth range: {wealth_index.min():.3f} – {wealth_index.max():.3f}")
print(f"NTL–wealth r:     {np.corrcoef(ntl, wealth_index)[0,1]:.3f}  (literature: ~0.65)")

## Part 2 — Stage 1: CNN features predict nighttime lights

In the real pipeline the CNN is trained end-to-end to minimise MSE(predicted NTL, actual NTL).
Here we verify that our synthetic features are informative by measuring how well a
linear probe (Ridge) can recover NTL from them — this mirrors the Stage 1 training loss.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(image_features)
y_ntl = ntl

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ntl_preds = np.zeros(N_TILES)
for train_idx, val_idx in kf.split(X):
    ridge = Ridge(alpha=1.0)
    ridge.fit(X[train_idx], y_ntl[train_idx])
    ntl_preds[val_idx] = ridge.predict(X[val_idx])

r2_ntl = r2_score(y_ntl, ntl_preds)
r_ntl  = np.corrcoef(y_ntl, ntl_preds)[0, 1]
print(f"Stage 1 — NTL prediction from CNN features (5-fold CV):")
print(f"  R² = {r2_ntl:.3f}   (Jean et al. report R²≈0.75 with full CNN training)")
print(f"  r  = {r_ntl:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(y_ntl, ntl_preds, alpha=0.5, s=18, c=wealth_index, cmap='RdYlGn', vmin=0, vmax=1)
ax.plot([y_ntl.min(), y_ntl.max()], [y_ntl.min(), y_ntl.max()], 'k--', lw=1)
ax.set_xlabel('Actual VIIRS NTL (nW/cm²/sr)')
ax.set_ylabel('CNN-predicted NTL')
ax.set_title(f'Stage 1: CNN features → NTL\nr = {r_ntl:.3f}  R² = {r2_ntl:.3f}')
plt.colorbar(sc, ax=ax, label='DHS wealth index (colour)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage1_ntl_prediction.png', dpi=150)
plt.show()
print('Saved stage1_ntl_prediction.png')

## Part 3 — Stage 2: CNN features → DHS wealth index

This is the transfer learning payoff. Features trained on NTL transfer to predict
survey-based wealth because both reflect the same economic geography:
roads, building density, roof materials, agricultural patterns.

**Why Ridge?** DHS cluster counts (~300–600/country) are small relative to feature
dimensionality (512), so L2 regularisation prevents overfitting.

**Jean et al. key result**: r² ≈ 0.68 generalising to *held-out countries*,
demonstrating true out-of-distribution generalisation.

In [ ]:
y_wealth = wealth_index

wealth_preds = np.zeros(N_TILES)
for train_idx, val_idx in kf.split(X):
    ridge = Ridge(alpha=10.0)
    ridge.fit(X[train_idx], y_wealth[train_idx])
    wealth_preds[val_idx] = ridge.predict(X[val_idx])

r2_wealth = r2_score(y_wealth, wealth_preds)
r_wealth  = np.corrcoef(y_wealth, wealth_preds)[0, 1]
print(f"Stage 2 — DHS wealth prediction from CNN features (5-fold CV):")
print(f"  R² = {r2_wealth:.3f}   (Jean et al. report R²≈0.68 cross-country)")
print(f"  r  = {r_wealth:.3f}")

# Dense poverty map: predict wealth for ALL grid locations (no ground truth needed)
GRID = 40
lon_grid = np.linspace(LON_MIN, LON_MAX, GRID)
lat_grid = np.linspace(LAT_MIN, LAT_MAX, GRID)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

grid_ntl = np.zeros(GRID * GRID)
for city_lon, city_lat, peak in CITIES:
    dist = np.sqrt((lon_mesh.ravel() - city_lon)**2 + (lat_mesh.ravel() - city_lat)**2)
    grid_ntl += peak * np.exp(-dist**2 / (1.5**2))
grid_ntl = np.clip(grid_ntl, 0, 80)

grid_features = (
    np.outer(grid_ntl / grid_ntl.max(), urban_direction) * 3.0
    + np.random.randn(GRID*GRID, FEAT_DIM) * 1.0
)
grid_X = scaler.transform(grid_features)

ridge_final = Ridge(alpha=10.0).fit(X, y_wealth)
grid_wealth = ridge_final.predict(grid_X).reshape(GRID, GRID)
print(f"\nPoverty map coverage: {GRID}×{GRID} tiles across Nigeria bbox")

## Part 4 — Full pipeline visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    'Jean et al. (2016) Transfer Learning Pipeline — Nigeria region (synthetic demo)\n'
    'Stage 1: CNN features → NTL  |  Stage 2: CNN features → DHS wealth  |  Poverty map',
    fontsize=11, fontweight='bold'
)

# Panel 1: Stage 2 scatter
sc0 = axes[0].scatter(y_wealth, wealth_preds, alpha=0.5, s=15,
                      c=ntl, cmap='inferno', vmin=0, vmax=80)
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('Actual DHS Wealth Index')
axes[0].set_ylabel('Predicted Wealth Index')
axes[0].set_title(f'Stage 2: CNN Features → Wealth\nr = {r_wealth:.3f}  R² = {r2_wealth:.3f}')
plt.colorbar(sc0, ax=axes[0], label='VIIRS NTL intensity')

# Panel 2: DHS cluster map (ground truth)
sc1 = axes[1].scatter(lons, lats, c=wealth_index, cmap='RdYlGn',
                      s=20, alpha=0.8, vmin=0, vmax=1)
for city_lon, city_lat, _ in CITIES:
    axes[1].plot(city_lon, city_lat, 'k*', ms=14)
axes[1].set_xlabel('Longitude (°E)')
axes[1].set_ylabel('Latitude (°N)')
axes[1].set_title('DHS Survey Clusters (ground truth)\n★ = Lagos, Kano, Abuja')
plt.colorbar(sc1, ax=axes[1], label='DHS Wealth Index')

# Panel 3: Dense poverty map
im = axes[2].imshow(
    grid_wealth, origin='lower', cmap='RdYlGn',
    extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    aspect='auto', vmin=0, vmax=1
)
for city_lon, city_lat, _ in CITIES:
    axes[2].plot(city_lon, city_lat, 'k*', ms=14)
axes[2].set_xlabel('Longitude (°E)')
axes[2].set_ylabel('Latitude (°N)')
axes[2].set_title('Predicted Poverty Map (dense grid)\nGreen = wealthier  |  Red = poorer')
plt.colorbar(im, ax=axes[2], label='Predicted Wealth Index')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'jean2016_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved jean2016_pipeline.png')

## Part 5 — Connecting to the Chalmers PhD research agenda

### How this pipeline extends to the three PhD objectives

**Objective 1 — Multidimensional poverty from Sentinel-2 over Africa**

The notebook above shows single-year, single-country, consumption-poverty estimation.
The PhD extends this along three dimensions:

| Dimension | This notebook | PhD extension |
|---|---|---|
| Target | DHS wealth index (unidimensional) | Multidimensional Poverty Index (MPI) |
| Time | Single year | Annual composites 2014–2024 → poverty trajectories |
| Scale | One country (Nigeria) | 20+ Sub-Saharan African countries |
| Model | Ridge on CNN features | Possibly LSTM/Transformer for temporal, GP for spatial uncertainty |

**Objective 2 — Comparing satellite resolutions**

The Stage 1 CNN is the only component that changes between resolutions.
This codebase's preprocessing pipeline (`src/data/preprocessing.py`) already handles
multi-resolution rasters via `rasterio` — swapping Sentinel-2 for Pléiades or Landsat
requires only changing the input path and spatial aggregation window.

```
Resolution   Pixels per 10km DHS buffer   Data cost/country/year
Pléiades 2m     ~25,000,000                Very high (commercial)
Sentinel-2 10m  ~1,000,000                 Free (Copernicus Open Access)
Landsat 30m     ~110,000                   Free (USGS Earth Explorer)
```

**Objective 3 — XAI for policy trust**

GradCAM is already implemented in this repo (`/explain` API endpoint, Notebook 04).
Applied to the Stage 1 CNN over African imagery, GradCAM can answer:

> *"Does the model attend to roof materials (durable vs. thatch) — a controllable housing
> policy target — or to geographic features like proximity to roads, which reflect
> structural inequality?"*

This distinction is policy-critical: interventions should target the features
the model is actually sensitive to.

### References
- Jean, N., Burke, M., Xie, M., Davis, W. M., Lobell, D. B., & Ermon, S. (2016).
  Combining satellite imagery and machine learning to predict poverty. *Science*, 353(6301), 790–794.
- Yeh, C., Perez, A., Driscoll, A., Azzari, G., Tang, Z., Lobell, D., ... & Burke, M. (2020).
  Using publicly available satellite imagery and deep learning to understand economic well-being in Africa.
  *Nature Communications*, 11, 2583.
- Henderson, J. V., Storeygard, A., & Weil, D. N. (2012).
  Measuring economic growth from outer space. *American Economic Review*, 102(2), 994–1028.
- Engstrom, R., Hersh, J., & Newhouse, D. (2017).
  Poverty from space: using high-resolution satellite imagery for estimating economic well-being.
  *World Bank Policy Research Working Paper 8284*.